# Ground-Truth Review & Export

Upload a **slim Excel** (`query`, `actual_output`/`expected_output`, `extracted_metadata`),
review the rows (Arabic-aware, RTL), tick the ones that are good ground truth, and click
**Generate CSV** to produce a UTF-8 file with columns `query`, `expected_output`, `metadata`
— ready to upload as responses with ground-truth data.

**How to use**
1. Run cell 1 (setup).
2. Run cell 2 and either use the file picker or set `EXCEL_PATH` and run cell 3.
3. Review rows in cell 4, tick the good ones.
4. Run cell 5, click **Generate CSV**, download from the printed path.

The answer column (`actual_output`, or `expected_output` if the slim file already renamed it)
becomes **`expected_output`** in the output. CSV is written as **UTF-8 with BOM** so Arabic
opens correctly in Excel.

## 1. Setup

In [ ]:
# %pip install ipywidgets pandas openpyxl --quiet
import pandas as pd, csv, datetime as dt
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

OUT_DIR = Path("ground_truth_exports"); OUT_DIR.mkdir(exist_ok=True)

def _detect_columns(df):
    """Map source columns -> (query, answer, metadata), tolerant of slim or raw names."""
    cols = {c.strip().lower(): c for c in df.columns}
    qcol = cols.get("query") or cols.get("input")
    acol = cols.get("actual_output") or cols.get("expected_output") or cols.get("answer")
    mcol = cols.get("extracted_metadata") or cols.get("metadata")
    if qcol is None:  # fallback: first column
        qcol = list(df.columns)[0]
    return qcol, acol, mcol

print("Setup ready. Exports ->", OUT_DIR.resolve())

## 2. Choose the Excel file

Either use the upload widget below (drag/drop or browse), **or** skip it and set `EXCEL_PATH`
manually in cell 3.

In [ ]:
uploader = widgets.FileUpload(accept=".xlsx", multiple=False, description="Upload slim .xlsx")
display(uploader)
print("After picking a file here, run cell 3. (Or ignore this and set EXCEL_PATH in cell 3.)")

## 3. Load the rows

In [ ]:
EXCEL_PATH = ""   # optional: set to a path like "langfuse_exports/xxx_slim.xlsx" to load directly

df = None
if EXCEL_PATH:
    df = pd.read_excel(EXCEL_PATH)
    print("Loaded from EXCEL_PATH:", EXCEL_PATH)
elif uploader.value:
    # ipywidgets v8: .value is a tuple of dicts with 'content'
    up = uploader.value[0] if isinstance(uploader.value, (list, tuple)) else list(uploader.value.values())[0]
    content = up["content"] if isinstance(up, dict) else up.content
    import io
    df = pd.read_excel(io.BytesIO(content))
    name = up.get("name") if isinstance(up, dict) else getattr(up, "name", "uploaded.xlsx")
    print("Loaded from upload:", name)
else:
    raise ValueError("No file. Use the uploader in cell 2, or set EXCEL_PATH in this cell.")

df = df.fillna("")
QCOL, ACOL, MCOL = _detect_columns(df)
print(f"Rows: {len(df)}")
print(f"Detected columns -> query: {QCOL!r} | answer: {ACOL!r} | metadata: {MCOL!r}")
if ACOL is None:
    print("WARNING: no answer column found (expected actual_output / expected_output).")
df.head(3)

## 4. Review & select rows

Each row shows the query, the answer (which becomes `expected_output`), and the circular
metadata, rendered right-to-left for Arabic. Tick the rows that are good ground truth.

Use **Select all / Clear all** for speed. Selection is held in memory until you export.

In [ ]:
row_checks = []

def _rtl(text):
    safe = (str(text) if text is not None else "").replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
    return (f"<div dir='rtl' style='text-align:right; white-space:pre-wrap; "
            f"font-family:\"Segoe UI\",Tahoma,Arial,sans-serif; line-height:1.6;'>{safe}</div>")

def build_review():
    global row_checks
    row_checks = []
    blocks = []
    for i in range(len(df)):
        q = df.iloc[i][QCOL]
        a = df.iloc[i][ACOL] if ACOL else ""
        m = df.iloc[i][MCOL] if MCOL else ""
        chk = widgets.Checkbox(value=False, description=f"Use row {i}", indent=False)
        row_checks.append(chk)
        header = widgets.HTML(f"<hr><b>Row {i}</b>")
        qh = widgets.HTML("<b>Query</b>" + _rtl(q))
        ah = widgets.HTML("<b>Answer &rarr; expected_output</b>" + _rtl(a))
        mh = widgets.HTML("<b>Metadata</b>" + _rtl(m)) if MCOL else widgets.HTML("")
        blocks.append(widgets.VBox([header, chk, qh, ah, mh],
                                   layout=widgets.Layout(border="1px solid #e0e0e0",
                                                         padding="8px", margin="4px 0")))
    sel_all = widgets.Button(description="Select all", button_style="success")
    clr_all = widgets.Button(description="Clear all")
    counter = widgets.HTML()
    def _refresh(_=None):
        n = sum(c.value for c in row_checks)
        counter.value = f"<b>{n}</b> / {len(row_checks)} selected"
    def _sel(_): 
        for c in row_checks: c.value = True
    def _clr(_):
        for c in row_checks: c.value = False
    sel_all.on_click(_sel); clr_all.on_click(_clr)
    for c in row_checks: c.observe(_refresh, names="value")
    _refresh()
    display(widgets.HBox([sel_all, clr_all, counter]))
    display(widgets.VBox(blocks))

build_review()

## 5. Generate the ground-truth CSV

In [ ]:
gen_btn = widgets.Button(description="Generate CSV", button_style="primary",
                         icon="download", layout=widgets.Layout(width="200px"))
gen_out = widgets.Output()

def _generate(_):
    with gen_out:
        clear_output()
        idx = [i for i, c in enumerate(row_checks) if c.value]
        if not idx:
            print("No rows selected. Tick at least one row in cell 4.")
            return
        out = df.iloc[idx].copy()
        rename = {QCOL: "query"}
        if ACOL: rename[ACOL] = "expected_output"
        if MCOL: rename[MCOL] = "metadata"
        out = out.rename(columns=rename)
        keep = [c for c in ["query", "expected_output", "metadata"] if c in out.columns]
        out = out[keep]

        stamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
        path = OUT_DIR / f"ground_truth_{stamp}.csv"
        # UTF-8 with BOM so Excel renders Arabic correctly; QUOTE_MINIMAL keeps newlines in cells safe
        out.to_csv(path, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_MINIMAL)
        print(f"Wrote {len(out)} row(s) -> {path}")
        print("Columns:", list(out.columns))
        display(out.head(min(10, len(out))))

gen_btn.on_click(_generate)
display(gen_btn, gen_out)

---
### Notes
- Output columns: **`query`**, **`expected_output`** (from the answer), **`metadata`**
  (from `extracted_metadata`). Missing source columns are simply omitted.
- Encoding is **UTF-8 with BOM** (`utf-8-sig`) — the reliable way to make Arabic open
  correctly when the CSV is double-clicked in Excel.
- Cell newlines inside `expected_output`/`metadata` are preserved and properly quoted, so the
  multi-line `extracted_metadata` stays in one cell.
- Re-running cell 4 resets selections. Selections persist while you move between cells 4 and 5.
- To re-import into Langfuse as a dataset, `query` maps to the item input and
  `expected_output` to the item's expected output; rename if your importer needs exact keys.